# `09_thesis_results`: Reproducing the thesis Results section

This notebook isolates the exact cells from `09_bicycle_route_vs_non_bicycle_route_high_level` that produce the figures, tables, and statistics reported in the thesis Results section. Running it end-to-end reproduces every numbered table and figure cited in the thesis, plus the Kruskal-Wallis, Dunn's post-hoc, and Moran's I results referenced in the prose.

## Inputs

All inputs are cached parquet files written by upstream pipeline notebooks:

- `bicycle_route_infrastructure_per_side_per_municipality` (from `06_bicycle_route_infrastructure_per_side_per_spatial_unit`): the per-side classification table that carries `unece_class`, `classifiability`, `evidence_basis`, `protection_level_per_side`, `degree_of_urbanisation`, `municipality_code`, `municipality_name`, `area_km2`, `geometry`, and `clipped_length_meters`. This is the load-bearing input for every Results section.
- `municipalities`, `provinces` (from `03_boundaries_population`): boundary geometries for choropleths and LISA contiguity.

## Outputs reproduced

| Thesis reference | Section |
|---|---|
| Table 5 (national classification outcomes: 52.2% / 47.3% / 0.5%) | §3 |
| Table 6 (urban-rural gradient: 73.94% → 44.83% classifiable) | §4 |
| Figure 7a (boxplot of classifiability by urban class, Wormerland outlier) | §4 |
| Figure 7b (boxplot of network length by urban class) | §4 |
| Kruskal-Wallis H = 74.67, p < 0.001 and Dunn's Holm post-hoc | §5 |
| Figure 6 (choropleth of `pct_classifiable` per municipality) | §6 |
| Moran's I = 0.45 (p < 0.001) and Figure 8 (LISA clusters: 52 HH, 58 LL, 7 HL, 4 LH) | §7 |
| Figure 9 (decomposition of classifiability by design / via tags / unclassifiable, plus exposure and completeness) | §8 |
| Table 9 (highway-type composition by urbanisation) | §9 |
| Discussion: rcn node-network gradient 71.6% → 41.8% | §10 |
| Figure 10 (composition of unclassifiable share: `inferred_absence_none` vs `inferred_absence_partial`) | §11 |

## Dependencies on prior notebooks

- `03_boundaries_population.ipynb` and `functions.ipynb` via `%run`.
- All `06_*` per-spatial-unit outputs and `07_*` metric outputs as cached parquets.

## Table of contents

1. [Environment setup](#1-environment-setup)
2. [Per-municipality and national quality tables](#2-per-municipality-and-national-quality-tables)
3. [Table 5: national classification outcomes](#3-table-5)
4. [Table 6 + Figure 7: urban-rural gradient](#4-table-6-and-figure-7)
5. [Kruskal-Wallis and Dunn's post-hoc](#5-kruskal-wallis-and-dunns-post-hoc)
6. [Figure 6: choropleth of `pct_classifiable`](#6-figure-6)
7. [Spatial clustering: Moran's I and Figure 8 LISA](#7-spatial-clustering)
8. [Figure 9: decomposition by design vs via tags](#8-figure-9)
9. [Table 9: highway-type composition by urbanisation](#9-table-9)
10. [Discussion: rcn node-network gradient](#10-discussion-rcn-node-network)
11. [Figure 10: composition of unclassifiable share](#11-figure-10)

---

## 1. Environment setup

### Libraries and extensions

In [5]:
import contextily as ctx
import duckdb
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from esda.moran import Moran, Moran_Local
from IPython.display import display
from IPython.utils import io
from libpysal.weights import Queen
from matplotlib_scalebar.scalebar import ScaleBar
from pathlib import Path
from scipy.stats import kruskal
from shapely import wkt
import scikit_posthocs as sp

### Loading shared variables

The `municipalities` and `provinces` GeoDataFrames are brought in via `%run` of `03_boundaries_population` (the choropleth and LISA sections need `provinces` for the overlay and `municipalities` for the categorical urbanisation order). The per-side per-municipality table that carries every column the Results section depends on is loaded directly from the cached parquet output of `06_bicycle_route_infrastructure_per_side_per_spatial_unit`.

In [6]:
with io.capture_output() as captured:
    %run 03_boundaries_population.ipynb
    %run functions.ipynb

bicycle_route_infrastructure_per_side_per_municipality = duckdb.read_parquet(
    "/local/data/vbo226/cache/bicycle_route_infrastructure_per_side_per_municipality.parquet"
)

Define the ordinal urbanisation order once. It is reused by every per-urbanisation table, boxplot, and SQL query below.

In [ ]:
urban_order = [
    "Very highly urbanized",
    "Highly urbanized",
    "Moderately urbanized",
    "Slightly urbanized",
    "Non-urbanized",
]

municipalities['degree_of_urbanisation'] = pd.Categorical(
    municipalities['degree_of_urbanisation'],
    categories=urban_order,
    ordered=True,
)

---

## 2. Per-municipality and national quality tables

Two aggregations are computed from `bicycle_route_infrastructure_per_side_per_municipality`. Both partition every side-row into one of six mutually-exclusive infrastructure classes (`cycle_track`, `cycle_lane`, `cycle_street`, `mixed_traffic` classifiable, `mixed_traffic` unclassifiable, `excluded`), and both derive `classifiable_km` as the rollup of the four classifiable buckets. The per-municipality table additionally carries the `degree_of_urbanisation` and the WKB geometry, since every subsequent table, figure, and spatial statistic in the Results section is driven from it.

### 2.1 `municipality_infrastructure_quality`

In [ ]:
municipality_infrastructure_quality = duckdb.sql("""
WITH agg AS (
    SELECT
        municipality_code,
        any_value(municipality_name)      AS municipality_name,
        any_value(degree_of_urbanisation) AS degree_of_urbanisation,
        any_value(area_km2)               AS municipality_area_km2,
        geometry,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'cycle_track'
        ), 0) / 1000.0 AS track_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'cycle_lane'
        ), 0) / 1000.0 AS lane_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'cycle_street'
        ), 0) / 1000.0 AS street_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'mixed_traffic'
            AND classifiability = 'classifiable'
        ), 0) / 1000.0 AS mixed_certain_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'mixed_traffic'
            AND classifiability = 'unclassifiable'
        ), 0) / 1000.0 AS mixed_uncertain_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE classifiability = 'excluded'
        ), 0) / 1000.0 AS excluded_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE evidence_basis = 'recorded_absence_sidepath'
        ), 0) / 1000.0 AS sidepath_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE evidence_basis = 'inferred_absence_none'
        ), 0) / 1000.0 AS inferred_absence_none_km,

        COALESCE(SUM(clipped_length_meters) FILTER (
            WHERE evidence_basis = 'inferred_absence_partial'
        ), 0) / 1000.0 AS inferred_absence_partial_km,

        COALESCE(SUM(clipped_length_meters), 0) / 1000.0 AS total_km
    FROM bicycle_route_infrastructure_per_side_per_municipality
    GROUP BY municipality_code, geometry
),
derived AS (
    SELECT *,
        (track_km + lane_km + street_km + mixed_certain_km) AS classifiable_km,
        (inferred_absence_none_km + inferred_absence_partial_km) AS inferred_absence_km
    FROM agg
)
SELECT
    municipality_code,
    municipality_name,
    degree_of_urbanisation,
    municipality_area_km2,
    geometry,

    total_km,

    classifiable_km,
    mixed_uncertain_km,
    excluded_km,

    100.0 * classifiable_km    / NULLIF(total_km, 0) AS pct_classifiable,
    100.0 * mixed_uncertain_km / NULLIF(total_km, 0) AS pct_unclassifiable,
    100.0 * excluded_km        / NULLIF(total_km, 0) AS pct_excluded,

    track_km,
    lane_km,
    street_km,
    mixed_certain_km,

    100.0 * track_km         / NULLIF(classifiable_km, 0) AS pct_track_of_classifiable,
    100.0 * lane_km          / NULLIF(classifiable_km, 0) AS pct_lane_of_classifiable,
    100.0 * street_km        / NULLIF(classifiable_km, 0) AS pct_street_of_classifiable,
    100.0 * mixed_certain_km / NULLIF(classifiable_km, 0) AS pct_mixed_certain_of_classifiable,

    (track_km + lane_km) AS facility_km,
    100.0 * track_km / NULLIF(track_km + lane_km, 0) AS pct_track_of_facility,

    inferred_absence_none_km,
    inferred_absence_partial_km,
    inferred_absence_km,

    100.0 * inferred_absence_km / NULLIF(classifiable_km, 0)
        AS pct_inferred_absence_of_classifiable,

    sidepath_km
FROM derived
ORDER BY municipality_code
""").df()

_parts = ["track_km", "lane_km", "street_km", "mixed_certain_km", "mixed_uncertain_km", "excluded_km"]

assert (
    municipality_infrastructure_quality[_parts].sum(axis=1)
    - municipality_infrastructure_quality["total_km"]
).abs().max() < 0.05

assert (
    municipality_infrastructure_quality["mixed_uncertain_km"]
    - (
        municipality_infrastructure_quality["inferred_absence_none_km"]
        + municipality_infrastructure_quality["inferred_absence_partial_km"]
    )
).abs().max() < 0.05

assert (
    municipality_infrastructure_quality["classifiable_km"]
    - (
        municipality_infrastructure_quality["track_km"]
        + municipality_infrastructure_quality["lane_km"]
        + municipality_infrastructure_quality["street_km"]
        + municipality_infrastructure_quality["mixed_certain_km"]
    )
).abs().max() < 0.05

assert municipality_infrastructure_quality["degree_of_urbanisation"].isna().sum() == 0

municipality_infrastructure_quality.head()

### 2.2 `national_infrastructure_quality`

The national table sums the same partition without grouping by municipality. It is the direct source of the headline figures in Table 5 of the thesis.

In [ ]:
national_infrastructure_quality = duckdb.sql("""
WITH national AS (
    SELECT
        SUM(clipped_length_meters) / 1000.0 AS total_km,

        SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_track') / 1000.0 AS track_km,
        SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_lane') / 1000.0 AS lane_km,
        SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_street') / 1000.0 AS street_km,

        SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'mixed_traffic'
            AND classifiability = 'classifiable'
        ) / 1000.0 AS mixed_certain_km,

        SUM(clipped_length_meters) FILTER (
            WHERE unece_class = 'mixed_traffic'
            AND classifiability = 'unclassifiable'
        ) / 1000.0 AS mixed_uncertain_km,

        SUM(clipped_length_meters) FILTER (
            WHERE classifiability = 'excluded'
        ) / 1000.0 AS excluded_km,

        (
            SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_track')
            + SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_lane')
            + SUM(clipped_length_meters) FILTER (WHERE unece_class = 'cycle_street')
            + SUM(clipped_length_meters) FILTER (
                WHERE unece_class = 'mixed_traffic'
                AND classifiability = 'classifiable'
            )
        ) / 1000.0 AS classifiable_km,

        SUM(clipped_length_meters) FILTER (
            WHERE evidence_basis = 'inferred_absence_none'
        ) / 1000.0 AS inferred_absence_none_km,

        SUM(clipped_length_meters) FILTER (
            WHERE evidence_basis = 'inferred_absence_partial'
        ) / 1000.0 AS inferred_absence_partial_km
    FROM bicycle_route_infrastructure_per_side_per_municipality
)
SELECT
    total_km,

    classifiable_km,
    mixed_uncertain_km,
    excluded_km,

    100.0 * classifiable_km    / NULLIF(total_km, 0) AS pct_classifiable,
    100.0 * mixed_uncertain_km / NULLIF(total_km, 0) AS pct_unclassifiable,
    100.0 * excluded_km        / NULLIF(total_km, 0) AS pct_excluded,

    track_km,
    lane_km,
    street_km,
    mixed_certain_km,

    100.0 * track_km         / NULLIF(classifiable_km, 0) AS pct_track_of_classifiable,
    100.0 * lane_km          / NULLIF(classifiable_km, 0) AS pct_lane_of_classifiable,
    100.0 * street_km        / NULLIF(classifiable_km, 0) AS pct_street_of_classifiable,
    100.0 * mixed_certain_km / NULLIF(classifiable_km, 0) AS pct_mixed_certain_of_classifiable,

    (track_km + lane_km) AS facility_km,
    100.0 * track_km / NULLIF(track_km + lane_km, 0) AS pct_track_of_facility,

    inferred_absence_none_km,
    inferred_absence_partial_km,
    (inferred_absence_none_km + inferred_absence_partial_km) AS inferred_absence_km,

    100.0 * (inferred_absence_none_km + inferred_absence_partial_km)
        / NULLIF(classifiable_km, 0) AS pct_inferred_absence_of_classifiable
FROM national
""").df()

---

## 3. Table 5: national classification outcomes

Reproduces the thesis-headline figures: 41,814.7 km total, 52.2% classifiable, 47.3% unclassifiable, 0.5% excluded; and the within-classifiable composition (cycle tracks 70.4%, cycle lanes 17.1%, mixed traffic 11.0%, cycle streets 1.5%).

In [ ]:
pd.set_option('display.max_columns', None)
national_infrastructure_quality

---

## 4. Table 6 and Figure 7: urban-rural gradient

### 4.1 Table 6 source: `stats_by_urban_class`

Median `pct_classifiable` and median `total_km` per urbanisation class, plus the count of municipalities in each class.

In [ ]:
quality_cols = [
    "pct_classifiable",
    "pct_track_of_classifiable",
    "pct_mixed_certain_of_classifiable",
    "pct_lane_of_classifiable",
    "pct_street_of_classifiable",
    "total_km",
]

stats_by_urban_class = (
    municipality_infrastructure_quality
    .groupby("degree_of_urbanisation", observed=True)[quality_cols]
    .agg(["count", "median"])
    .reindex(urban_order)
)
stats_by_urban_class

In [ ]:
### 4.2 Figure 7a: boxplot of `pct_classifiable` by urban class

Outliers labelled by municipality name. The thesis singles out Wormerland as the labelled outlier in the *Highly urbanized* class.

In [ ]:
df = municipality_infrastructure_quality.melt(
    id_vars=["municipality_name", "degree_of_urbanisation"],
    value_vars=quality_cols,
    var_name="metric",
    value_name="value",
)

df = df[df["metric"] == "pct_classifiable"]

g = sns.catplot(
    data=df,
    x="value",
    y="degree_of_urbanisation",
    order=urban_order,
    kind="box",
    color="lightgray",
    height=3.5,
    aspect=1.3,
    showfliers=True,
)

ax = g.ax
y_map = {k: i for i, k in enumerate(urban_order)}
offsets = [(-14, 6), (14, 6), (0, 15)]

for level in urban_order:
    sub = df[df["degree_of_urbanisation"] == level]
    vals = sub["value"].dropna()

    q1 = vals.quantile(0.25)
    q3 = vals.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = sub[(sub["value"] < lower) | (sub["value"] > upper)].copy()
    y = y_map[level]

    outliers = outliers.sort_values("value")

    for i, (_, row) in enumerate(outliers.iterrows()):
        dx, dy = offsets[i % len(offsets)]
        ax.annotate(
            row["municipality_name"],
            (row["value"], y),
            textcoords="offset points",
            xytext=(dx, dy),
            ha="center",
            va="bottom",
            fontsize=8,
            color="grey",
        )

g.fig.suptitle("Pct classifiable by degree of urbanisation", y=1.02)
g.set_axis_labels("Value", "Degree of urbanisation")
g.set_titles("")
sns.despine()
plt.tight_layout()
plt.show()

### 4.3 Figure 7b: boxplot of network length by urban class

Same encoding as 7a but for `total_km` (the per-municipality bicycle-network length). Median network length rises from approximately 68 km in very-highly urbanised municipalities to 177.6 km in non-urbanised municipalities.

In [ ]:
df_length = municipality_infrastructure_quality[
    ["municipality_name", "degree_of_urbanisation", "total_km"]
].copy()

g = sns.catplot(
    data=df_length,
    x="total_km",
    y="degree_of_urbanisation",
    order=urban_order,
    kind="box",
    color="lightgray",
    height=3.5,
    aspect=1.3,
    showfliers=True,
)

g.fig.suptitle("Total network length (km) by degree of urbanisation", y=1.02)
g.set_axis_labels("Total km", "Degree of urbanisation")
g.set_titles("")
sns.despine()
plt.tight_layout()
plt.show()

---

## 5. Kruskal-Wallis and Dunn's post-hoc

Tests whether `pct_classifiable` differs significantly across urbanisation classes. The thesis reports H = 74.67, p < 0.001 with monotonically increasing mean ranks.

In [ ]:
groups = [
    group["pct_classifiable"].dropna().values
    for _, group in municipality_infrastructure_quality.groupby("degree_of_urbanisation", observed=True)
]

H, p = kruskal(*groups)

print("Kruskal-Wallis test")
print(f"H = {H:.4f}, p = {p:.6f}")

df = municipality_infrastructure_quality.copy()
df = df.dropna(subset=["pct_classifiable"])
df["rank"] = df["pct_classifiable"].rank(method="average")

mean_ranks = df.groupby("degree_of_urbanisation", observed=True)["rank"].mean()

print("\nMean ranks per group:")
print(mean_ranks.sort_values())

In [ ]:
posthoc = sp.posthoc_dunn(
    municipality_infrastructure_quality,
    val_col="pct_classifiable",
    group_col="degree_of_urbanisation",
    p_adjust="holm",
)
posthoc

---

## 6. Figure 6: choropleth of `pct_classifiable`

Per-municipality classifiability mapped over the Carto Positron basemap, with white municipality borders and black province borders overlaid. Colour ramp is `Blues` normalised to [0.15, 0.90] (matches the thesis figure).

In [ ]:
gdf = duckdb.sql("""
SELECT * REPLACE(ST_AsText(ST_GeomFromWKB(geometry)) AS geometry)
FROM municipality_infrastructure_quality
""").df()

gdf["geometry"] = gdf["geometry"].apply(wkt.loads)
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")

In [ ]:
gdf_track = gdf.copy()
gdf_track["vals"] = (gdf_track["pct_classifiable"].astype(float) / 100.0).fillna(0.0)

gdf_track = gdf_track.to_crs(epsg=3857)
provinces_gdf = provinces.to_crs(epsg=3857)

fig, ax = plt.subplots(1, 1, figsize=(12, 12), dpi=300)

cmap = plt.cm.Blues
norm = mpl.colors.Normalize(vmin=0.15, vmax=0.90)

gdf_track.plot(column="vals", cmap=cmap, norm=norm, linewidth=0, ax=ax, zorder=1)
gdf_track.boundary.plot(ax=ax, color="white", linewidth=0.35, zorder=2)
provinces_gdf.boundary.plot(ax=ax, color="black", linewidth=1.0, zorder=3)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.4, zorder=0)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, ax=ax, shrink=0.6)
cbar.set_label("Attribute completeness %")
cbar.ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

ax.set_title("PCT CLASSIFIABLE", fontsize=14, pad=15)

scalebar = ScaleBar(dx=1, units="m", location="lower right", box_alpha=0.6)
ax.add_artist(scalebar)

ax.annotate(
    "N",
    xy=(0.06, 0.92),
    xytext=(0.06, 0.80),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(facecolor="black", width=5, headwidth=15),
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold",
)

ax.set_axis_off()
plt.tight_layout()
plt.show()

---

## 7. Spatial clustering: Moran's I and Figure 8 LISA

Global Moran's I on `pct_classifiable` with Queen contiguity (Wadden islands are dropped automatically since they are island components in the contiguity graph). The thesis reports I = 0.45, p < 0.001, with 52 HH, 58 LL, 7 HL, and 4 LH municipalities at significance threshold 0.05.

In [ ]:
np.random.seed(42)

q_to_label = {
    1: "HH (High-High)",
    2: "LH (Low-High)",
    3: "LL (Low-Low)",
    4: "HL (High-Low)",
}

lisa_order = [
    "HH (High-High)",
    "LH (Low-High)",
    "LL (Low-Low)",
    "HL (High-Low)",
    "Not Significant",
]

cluster_colors_hex = {
    "HH (High-High)": "#d73027",
    "HL (High-Low)":  "#fdae61",
    "LH (Low-High)":  "#74add1",
    "LL (Low-Low)":   "#4575b4",
    "Not Significant": "#cccccc",
}

tag = "pct_classifiable"
label = "% classifiable"

sub = gdf.dropna(subset=[tag]).to_crs(3857).reset_index(drop=True).copy()

w = Queen.from_dataframe(sub, use_index=True)
w.transform = "r"
if w.islands:
    print(f"[warn] {label}: dropping {len(w.islands)} island(s)")
    sub = sub.drop(index=w.islands).reset_index(drop=True)
    w = Queen.from_dataframe(sub, use_index=True)
    w.transform = "r"

y = sub[tag].values
m_global = Moran(y, w, permutations=9999)
lisa = Moran_Local(y, w, permutations=9999, seed=42)

sub["lisa_type"] = np.where(
    lisa.p_sim < 0.05,
    pd.Series(lisa.q, index=sub.index).map(q_to_label),
    "Not Significant",
)
sub["lisa_type"] = pd.Categorical(sub["lisa_type"], categories=lisa_order, ordered=True)

print(f"Global Moran's I: {m_global.I:.4f}")
print(f"p-value (permutations): {m_global.p_sim:.4f}")
print(f"z-score: {m_global.z_sim:.4f}")
print()
print("LISA cluster counts:")
print(sub["lisa_type"].value_counts())

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 12), dpi=300)

sub.plot(
    color=sub["lisa_type"].map(cluster_colors_hex),
    linewidth=0,
    ax=ax,
    alpha=0.85,
    zorder=1,
)

sub.boundary.plot(ax=ax, color="white", linewidth=0.25, zorder=2)

ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, alpha=0.5, zorder=0)

handles = [
    mpl.patches.Patch(color=cluster_colors_hex[k], label=k)
    for k in lisa_order
]
ax.legend(handles=handles, loc="lower left", frameon=True, fontsize=10)

scalebar = ScaleBar(dx=1, units="m", location="lower right", box_alpha=0.6)
ax.add_artist(scalebar)

ax.annotate(
    "N",
    xy=(0.05, 0.95),
    xytext=(0.05, 0.85),
    xycoords="axes fraction",
    fontsize=14,
    ha="center",
    va="center",
    arrowprops=dict(facecolor="black", width=3, headwidth=10),
)

ax.set_title(f"LISA Clusters: {label}", fontsize=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

---

## 8. Figure 9: decomposition by design vs via tags vs unclassifiable

Splits the per-urbanisation total into three buckets: classifiable by design (`highway=cycleway`), classifiable via supplementary tags, and unclassifiable. Also computes the exposure share (everything not classifiable-by-design) and the completeness rate within the exposed portion. Matches the thesis Figure 9 table and the accompanying stacked bar chart.

In [ ]:
df_buckets = duckdb.sql("""
WITH classified AS (
    SELECT
        degree_of_urbanisation,
        clipped_length_meters AS len,
        CASE
            WHEN has_highway = 'cycleway' THEN 'classifiable_by_design'
            WHEN classifiability = 'classifiable' THEN 'classifiable_via_tags'
            ELSE 'unclassifiable'
        END AS bucket
    FROM bicycle_route_infrastructure_per_side_per_municipality
),
agg AS (
    SELECT
        degree_of_urbanisation,
        SUM(len) AS total_m,
        SUM(CASE WHEN bucket = 'classifiable_by_design' THEN len ELSE 0 END) AS by_design_m,
        SUM(CASE WHEN bucket = 'classifiable_via_tags'  THEN len ELSE 0 END) AS via_tags_m,
        SUM(CASE WHEN bucket = 'unclassifiable'         THEN len ELSE 0 END) AS unclass_m
    FROM classified
    GROUP BY degree_of_urbanisation
)
SELECT
    degree_of_urbanisation,
    ROUND(total_m / 1000, 1)              AS total_km,
    ROUND(100.0 * by_design_m / total_m, 1) AS pct_by_design,
    ROUND(100.0 * via_tags_m  / total_m, 1) AS pct_via_tags,
    ROUND(100.0 * unclass_m   / total_m, 1) AS pct_unclassifiable,
    ROUND(100.0 * (via_tags_m + unclass_m) / total_m, 1) AS pct_exposed,
    ROUND(100.0 * via_tags_m / NULLIF(via_tags_m + unclass_m, 0), 1) AS completeness_within_exposed
FROM agg
ORDER BY
    CASE degree_of_urbanisation
        WHEN 'Very highly urbanized' THEN 1
        WHEN 'Highly urbanized'      THEN 2
        WHEN 'Moderately urbanized'  THEN 3
        WHEN 'Slightly urbanized'    THEN 4
        WHEN 'Non-urbanized'         THEN 5
    END
""").df()

df_buckets

In [ ]:
labels = ['Very highly', 'Highly', 'Moderately', 'Slightly', 'Non-urbanized']
d = df_buckets.set_index('degree_of_urbanisation').reindex(urban_order)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {
    'pct_by_design':       '#2c7fb8',
    'pct_via_tags':        '#7fcdbb',
    'pct_unclassifiable':  '#d9d9d9',
}
legend = {
    'pct_by_design':       'Classifiable by design (highway=cycleway)',
    'pct_via_tags':        'Classifiable via supplementary tags',
    'pct_unclassifiable':  'Unclassifiable (missing tags)',
}

bottom = np.zeros(len(urban_order))
for col in ['pct_by_design', 'pct_via_tags', 'pct_unclassifiable']:
    ax.bar(labels, d[col].values, bottom=bottom, label=legend[col], color=colors[col])
    bottom += d[col].values

ax.set_ylabel('% of network length')
ax.set_xlabel('Degree of urbanisation')
ax.set_ylim(0, 100)
ax.legend(loc='lower left', fontsize=9)
fig.tight_layout()
plt.show()

---

## 9. Table 9: highway-type composition by urbanisation

For the four most prevalent highway types (`cycleway`, `unclassified`, `tertiary`, `residential`), reports total km, classifiability rate, and share of the urbanisation-class total. Matches the thesis Table 9 (also referred to as Table 7 in the prose).

In [ ]:
duckdb.sql("""
WITH base AS (
    SELECT
        degree_of_urbanisation,
        has_highway,
        SUM(clipped_length_meters) AS km,
        SUM(CASE WHEN classifiability = 'classifiable' THEN clipped_length_meters ELSE 0 END) AS classifiable_km
    FROM bicycle_route_infrastructure_per_side_per_municipality
    WHERE has_highway IN ('cycleway', 'unclassified', 'tertiary', 'residential')
    GROUP BY degree_of_urbanisation, has_highway
)
SELECT
    degree_of_urbanisation,
    has_highway AS highway,
    ROUND(km / 1000, 1) AS total_km,
    ROUND(100.0 * classifiable_km / NULLIF(km, 0), 1) AS pct_classifiable,
    ROUND(
        100.0 * km / SUM(km) OVER (PARTITION BY degree_of_urbanisation),
        1
    ) AS pct_within_urbanisation_class
FROM base
ORDER BY
    CASE degree_of_urbanisation
        WHEN 'Very highly urbanized' THEN 1
        WHEN 'Highly urbanized'      THEN 2
        WHEN 'Moderately urbanized'  THEN 3
        WHEN 'Slightly urbanized'    THEN 4
        WHEN 'Non-urbanized'         THEN 5
    END,
    pct_within_urbanisation_class DESC
""").df()

---

## 10. Discussion: rcn node-network gradient

Classifiability and cycleway share within each `way_m_bicycle_network_type_class` value (icn / ncn / rcn / lcn / multiple), broken down by urbanisation class. The thesis Discussion cites the rcn gradient (71.6% in very-highly urbanised → 41.8% in non-urbanised) to rule out a route-type explanation of the headline urban-rural gradient: the same gradient appears within the node-network layer that dominates the corpus.

In [ ]:
duckdb.sql("""
SELECT
    way_m_bicycle_network_type_class,
    degree_of_urbanisation,
    ROUND(SUM(clipped_length_meters) / 1000, 1) AS total_km,
    ROUND(
        100.0 * SUM(CASE WHEN classifiability = 'classifiable' THEN clipped_length_meters ELSE 0 END)
        / NULLIF(SUM(clipped_length_meters), 0),
        1
    ) AS pct_classifiable,
    ROUND(
        100.0 * SUM(CASE WHEN has_highway = 'cycleway' THEN clipped_length_meters ELSE 0 END)
        / NULLIF(SUM(clipped_length_meters), 0),
        1
    ) AS pct_cycleway
FROM bicycle_route_infrastructure_per_side_per_municipality
GROUP BY way_m_bicycle_network_type_class, degree_of_urbanisation
ORDER BY
    way_m_bicycle_network_type_class,
    CASE degree_of_urbanisation
        WHEN 'Very highly urbanized' THEN 1
        WHEN 'Highly urbanized'      THEN 2
        WHEN 'Moderately urbanized'  THEN 3
        WHEN 'Slightly urbanized'    THEN 4
        WHEN 'Non-urbanized'         THEN 5
    END
""").df()

---

## 11. Figure 10: composition of unclassifiable share

Splits the unclassifiable bucket (`classifiability = 'unclassifiable'`) into the two evidence bases used in the per-side framework: `inferred_absence_none` (neither side of a way carries cycleway tagging) and `inferred_absence_partial` (one side carries cycleway tagging, the other is left null). Reports kilometres and the share of each urban-class unclassifiable subtotal.

In [ ]:
df_fig10 = duckdb.sql("""
SELECT
    degree_of_urbanisation,
    evidence_basis,
    ROUND(SUM(clipped_length_meters) / 1000, 1) AS km,
    ROUND(
        100.0 * SUM(clipped_length_meters)
        / SUM(SUM(clipped_length_meters)) OVER (PARTITION BY degree_of_urbanisation),
        1
    ) AS pct_of_unclassifiable
FROM bicycle_route_infrastructure_per_side_per_municipality
WHERE classifiability = 'unclassifiable'
  AND evidence_basis IN ('inferred_absence_none', 'inferred_absence_partial')
GROUP BY degree_of_urbanisation, evidence_basis
ORDER BY
    CASE degree_of_urbanisation
        WHEN 'Very highly urbanized' THEN 1
        WHEN 'Highly urbanized'      THEN 2
        WHEN 'Moderately urbanized'  THEN 3
        WHEN 'Slightly urbanized'    THEN 4
        WHEN 'Non-urbanized'         THEN 5
    END,
    evidence_basis
""").df()

df_fig10

In [ ]:
pivot_fig10 = df_fig10.pivot(
    index='degree_of_urbanisation',
    columns='evidence_basis',
    values='pct_of_unclassifiable',
).reindex(urban_order)

labels = ['Very highly', 'Highly', 'Moderately', 'Slightly', 'Non-urbanized']

fig, ax = plt.subplots(figsize=(8, 5))

colors = {
    'inferred_absence_none':    '#fc8d59',
    'inferred_absence_partial': '#91bfdb',
}
legend = {
    'inferred_absence_none':    'inferred_absence_none (both sides null)',
    'inferred_absence_partial': 'inferred_absence_partial (one side tagged)',
}

bottom = np.zeros(len(urban_order))
for col in ['inferred_absence_none', 'inferred_absence_partial']:
    ax.bar(labels, pivot_fig10[col].values, bottom=bottom, label=legend[col], color=colors[col])
    bottom += pivot_fig10[col].values

ax.set_ylabel('% of unclassifiable share')
ax.set_xlabel('Degree of urbanisation')
ax.set_ylim(0, 100)
ax.legend(loc='lower left', fontsize=9)
ax.set_title('Composition of unclassifiable share by urbanisation class')
fig.tight_layout()
plt.show()